# Preprocess Audio files for Training
Computes a log-mel spectrogram for every 5s segment once and writes it to disk as `.npy`.
Run this **before** training. The training notebook reads the cache this produces.

In [10]:
# At the top of the Colab notebook, mount Drive if needed:
from google.colab import drive

import shutil, os
os.path.ismount('/content/drive') and drive.flush_and_unmount()
shutil.rmtree('/content/drive', ignore_errors=True)

# drive.flush_and_unmount()          # tear down any half-mount
drive.mount('/content/drive')

### NOTE #####
# Uncomment to Mount Google Drive (permissions needed)

Mounted at /content/drive


In [11]:
import os, random, warnings
import numpy as np
import pandas as pd
import ast
import librosa
from pathlib import Path
from sklearn.model_selection import GroupShuffleSplit

warnings.filterwarnings("ignore")

## 1. Config

In [13]:
CFG = {
    "seed": 42,
    "base": Path("/content/drive/MyDrive/Kaggle/birdclef-2026"),
    # audio (must match training notebook)
    "sr": 32000,
    "segment_sec": 5,
    "n_mels": 128,
    "n_fft": 2048,
    "hop_length": 512,
    "fmin": 20,
    "fmax": 16000,
    # split (must match training notebook)
    "val_fraction": 0.2,
    "cache_dir": Path("/content/drive/MyDrive/Kaggle/birdclef-2026/mel_cache"),
    "windows_per_clip": 5,
    "artifact_dir": Path("/content/drive/MyDrive/Kaggle/birdclef-2026/prep"),
}

CFG["artifact_dir"].mkdir(parents=True, exist_ok=True)
CFG["cache_dir"].mkdir(parents=True, exist_ok=True)
random.seed(CFG["seed"]); np.random.seed(CFG["seed"])

BASE                    = CFG["base"]
TRAIN_AUDIO             = BASE / "train_audio"
TRAIN_SOUNDSCAPES       = BASE / "train_soundscapes"
TRAIN_SOUNDSCAPE_LABELS = BASE / "train_soundscapes_labels.csv"
TRAIN_META              = BASE / "train.csv"
SAMPLE_SUB              = BASE / "sample_submission.csv"

## 2. Scored Species

In [14]:
# Scored species come from the submission columns, NOT from train metadata.
sample_sub = pd.read_csv(SAMPLE_SUB)
SPECIES = [c for c in sample_sub.columns if c != "row_id"]
NUM_CLASSES = len(SPECIES)
species_to_idx = {s: i for i, s in enumerate(SPECIES)}
print(f"{NUM_CLASSES} scored species")

234 scored species


## 3. Data Prep (same as training notebooks)

In [15]:
# ---- Training pool: XC/iNat clips (weak, one primary_label per file) ----
meta = pd.read_csv(TRAIN_META)
meta = meta[meta["primary_label"].isin(species_to_idx)].reset_index(drop=True)
meta["path"] = meta["filename"].map(lambda f: str(TRAIN_AUDIO / f))

def expand_windows(df, seg_len, n_win):
    rows = []
    for _, r in df.iterrows():
        try: dur = librosa.get_duration(path=r["path"])
        except Exception: dur = seg_len
        n = max(1, min(n_win, int(dur // seg_len)))
        for w in range(n):
            rr = r.copy(); rr["win_start"] = float(w * seg_len); rows.append(rr)
    return pd.DataFrame(rows).reset_index(drop=True)

meta_win = expand_windows(meta, CFG["segment_sec"], CFG["windows_per_clip"])

# ---- Soundscapes: strong per-segment multi-label; the only test-matched data ----
ss = pd.read_csv(TRAIN_SOUNDSCAPE_LABELS)
ss["path"] = ss["filename"].map(lambda f: str(TRAIN_SOUNDSCAPES / f))

def to_seconds(t):
    if isinstance(t, str) and ":" in t:
        h, m, s = t.split(":")
        return int(h)*3600 + int(m)*60 + float(s)
    return float(t)

ss["start"] = ss["start"].map(to_seconds)

# Split soundscapes by FILE (not segment) to avoid within-recording leakage.
gss = GroupShuffleSplit(n_splits=1, test_size=CFG["val_fraction"], random_state=CFG["seed"])
tr_idx, va_idx = next(gss.split(ss, groups=ss["filename"]))
ss_train, ss_val = ss.iloc[tr_idx].reset_index(drop=True), ss.iloc[va_idx].reset_index(drop=True)
print(f"train clips: {len(meta)} | soundscape segs -> train {len(ss_train)}  val {len(ss_val)}")

ss_train["kind"] = "sstr"; ss_val["kind"] = "ssva"
ss_split = pd.concat([ss_train, ss_val], ignore_index=True)
meta_win.to_csv(CFG["artifact_dir"] / "meta_windows.csv", index=True)
ss_split.to_csv(CFG["artifact_dir"] / "ss_split.csv", index=True)

train clips: 35549 | soundscape segs -> train 1182  val 296


## 4. Cache key + mel functions

In [16]:
# Cache location + key. Folder name encodes mel params so a param change -> fresh cache.
# NOTE: the split (seed, val_fraction) must match between the two notebooks, or the
# sstr_/ssva_ keys will not line up. The counts printed above are the alignment check.
# tag = f"m{CFG['n_mels']}_h{CFG['hop_length']}_f{CFG['fmax']}_s{CFG['segment_sec']}"
tag = f"m{CFG['n_mels']}_h{CFG['hop_length']}_f{CFG['fmax']}_s{CFG['segment_sec']}_w{CFG['windows_per_clip']}"
CACHE = CFG["cache_dir"] / tag
CACHE.mkdir(parents=True, exist_ok=True)

def cache_path(kind, idx):
    return CACHE / f"{kind}_{idx}.npy"

In [17]:
# def load_segment(path, sr, seg_len, offset=None, random_crop=False):
#     """Load a fixed-length waveform segment, pad/crop as needed."""
#     dur = librosa.get_duration(path=path)
#     if offset is None:
#         max_off = max(0.0, dur - seg_len)
#         offset = random.uniform(0, max_off) if random_crop else 0.0
#     y, _ = librosa.load(path, sr=sr, offset=offset, duration=seg_len)
#     target = int(sr * seg_len)
#     if len(y) < target:
#         y = np.pad(y, (0, target - len(y)))
#     return y[:target]

def load_segment(path, sr, seg_len, offset=0.0):
    y, _ = librosa.load(path, sr=sr, offset=offset, duration=seg_len)
    target = int(sr * seg_len)
    if len(y) < target:
        y = np.pad(y, (0, target - len(y)))
    return y[:target]

def wav_to_mel(y, cfg):
    m = librosa.feature.melspectrogram(
        y=y, sr=cfg["sr"], n_fft=cfg["n_fft"], hop_length=cfg["hop_length"],
        n_mels=cfg["n_mels"], fmin=cfg["fmin"], fmax=cfg["fmax"])
    m = librosa.power_to_db(m, ref=np.max)
    m = (m - m.mean()) / (m.std() + 1e-6)
    return m.astype(np.float32)

## 5. Precompute cache
Resumable — already-cached segments are skipped, so a disconnect just means rerun.

In [18]:
def precompute(df, kind, offset_col=None):
    done = 0
    for i, r in df.iterrows():
        p = cache_path(kind, i)
        if p.exists(): continue
        offset = float(r[offset_col]) if offset_col else None
        y = load_segment(r["path"], CFG["sr"], CFG["segment_sec"], offset=offset)
        np.save(p, wav_to_mel(y, CFG).astype(np.float16)); done += 1
    print(f"{kind}: {done} new")


clip_df = pd.read_csv(CFG["artifact_dir"] / "meta_windows.csv", index_col=0)
ss_df   = pd.read_csv(CFG["artifact_dir"] / "ss_split.csv", index_col=0)
sstr_df = ss_df[ss_df["kind"]=="sstr"].reset_index(drop=True)
ssva_df = ss_df[ss_df["kind"]=="ssva"].reset_index(drop=True)

precompute(clip_df, "clip", "win_start")
precompute(sstr_df, "sstr", "start")
precompute(ssva_df, "ssva", "start")


# precompute(meta,     "clip", offset_col="win_start")
# precompute(ss_train, "sstr", offset_col="start")
# precompute(ss_val,   "ssva", offset_col="start")
print("cache ready:", CACHE)

clip: 119333 new
sstr: 1182 new
ssva: 296 new
cache ready: /content/drive/MyDrive/Kaggle/birdclef-2026/mel_cache/m128_h512_f16000_s5_w5
